# Création du Gold test en JSONL (Few-shot)

Ce notebook génère le fichier **Gold_test** au format JSONL à partir de `Gold_test.json` et des CT dans `CT_json/`. Le résultat est sauvegardé dans ce dossier (`results/Fewshot/`) pour les expériences few-shot.

Deux formats de prompt sont proposés : **Prompt 1** (ancien, avec message system + PREMISE/HYPOTHESIS) et **Prompt 2** (nouveau, avec question « Is this premise in agreement... »).

## 1. Chemins et configuration

In [27]:
import json
from pathlib import Path

# Racine NLI4CT (depuis NLI4CT/ ou NLI4CT/results/Fewshot/)
cwd = Path(".").resolve()
if (cwd / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd
elif (cwd.parent.parent / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd.parent.parent
elif (cwd.parent / "Gold_test.json").exists():
    NLI4CT_ROOT = cwd.parent
else:
    NLI4CT_ROOT = cwd

GOLD_TEST_JSON = NLI4CT_ROOT / "Gold_test.json"
CT_JSON_DIR = NLI4CT_ROOT / "CT_json"
OUT_DIR = NLI4CT_ROOT / "results" / "Fewshot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("NLI4CT_ROOT:", NLI4CT_ROOT)
print("Gold_test.json:", GOLD_TEST_JSON, "->", GOLD_TEST_JSON.exists())
print("CT_json:", CT_JSON_DIR.exists())
print("Sortie:", OUT_DIR)

NLI4CT_ROOT: /Users/lubin/Documents/NLI_Finetuning/NLI4CT
Gold_test.json: /Users/lubin/Documents/NLI_Finetuning/NLI4CT/Gold_test.json -> True
CT_json: True
Sortie: /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot


In [28]:
# (Config effectuée dans la cellule précédente)

## 2. Fonctions de formatage (alignées sur create_dataset.ipynb)

In [29]:
def load_ct_data(ct_id):
    """Charge les données d'un essai clinique depuis CT_json/{ct_id}.json"""
    ct_file = CT_JSON_DIR / f"{ct_id}.json"
    if not ct_file.exists():
        return None
    with open(ct_file, "r", encoding="utf-8") as f:
        return json.load(f)


def extract_evidence(ct_data, section_id, evidence_indices):
    """Extrait les preuves d'une section selon les indices."""
    if ct_data is None:
        return ""
    section_data = ct_data.get(section_id, [])
    if not section_data:
        return ""
    evidence_items = []
    for idx in evidence_indices:
        if 0 <= idx < len(section_data):
            evidence_items.append(section_data[idx])
    return "\n".join(evidence_items)


# Prompt 2 (nouveau) : question explicite
def format_input_text_prompt2(statement, primary_evidence, secondary_evidence=None):
    if secondary_evidence:
        premise_content = f"{primary_evidence}\n\n{secondary_evidence}"
    else:
        premise_content = primary_evidence
    return (
        f"PREMISE: {premise_content}\n\n"
        f"Is this premise in agreement with the following hypothesis?\n"
        f"HYPOTHESIS: {statement}\n\n"
        f"Answer only with: Entailment or Contradiction."
    )


# Prompt 1 (ancien) : PREMISE + HYPOTHESIS, sans question
SYSTEM_MSG_PROMPT1 = "Classify the relationship between the premise and the hypothesis. Respond with only one word: 'Entailment' or 'Contradiction'."

# Pour le few-shot : instruction pour s'appuyer sur les exemples
SYSTEM_MSG_FEWSHOT = "You will see several examples of premise-hypothesis pairs with their classification (Entailment or Contradiction). Use these examples to understand the task. Then classify the relationship between the last premise and hypothesis. Respond with only one word: 'Entailment' or 'Contradiction'."

def format_input_text_prompt1(statement, primary_evidence, secondary_evidence=None):
    if secondary_evidence:
        premise_content = f"{primary_evidence}\n\n{secondary_evidence}"
    else:
        premise_content = primary_evidence
    return f"PREMISE: {premise_content}\n\nHYPOTHESIS: {statement}"

## 3. Génération du JSONL Gold test

In [30]:
def process_gold_test(format_fn, use_system_message=False):
    """Lit Gold_test.json, formate chaque entrée, retourne la liste et écrit le JSONL."""
    with open(GOLD_TEST_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    formatted = []
    errors = []

    for entry_id, entry in data.items():
        try:
            statement = entry.get("Statement", "")
            label = entry.get("Label", "")
            section_id = entry.get("Section_id", "")
            primary_id = entry.get("Primary_id", "")
            entry_type = entry.get("Type", "Single")

            primary_ct_data = load_ct_data(primary_id)
            if primary_ct_data is None:
                errors.append(f"{entry_id}: Primary CT {primary_id} not found")
                continue

            primary_evidence_indices = entry.get("Primary_evidence_index", [])
            primary_evidence = extract_evidence(primary_ct_data, section_id, primary_evidence_indices)

            secondary_evidence = None
            if entry_type == "Comparison":
                secondary_id = entry.get("Secondary_id", "")
                secondary_ct_data = load_ct_data(secondary_id)
                if secondary_ct_data is None:
                    errors.append(f"{entry_id}: Secondary CT {secondary_id} not found")
                    continue
                secondary_evidence_indices = entry.get("Secondary_evidence_index", [])
                secondary_evidence = extract_evidence(secondary_ct_data, section_id, secondary_evidence_indices)

            user_content = format_fn(statement, primary_evidence, secondary_evidence)

            if use_system_message:
                formatted_entry = {
                    "messages": [
                        {"role": "system", "content": SYSTEM_MSG_PROMPT1},
                        {"role": "user", "content": user_content},
                        {"role": "assistant", "content": label},
                    ]
                }
            else:
                formatted_entry = {
                    "messages": [
                        {"role": "user", "content": user_content},
                        {"role": "assistant", "content": label},
                    ]
                }
            formatted.append(formatted_entry)
        except Exception as e:
            errors.append(f"{entry_id}: {e}")

    return formatted, errors

In [31]:
# Générer les deux versions (Prompt 1 et Prompt 2) et les sauvegarder

for name, format_fn, use_system in [
    ("Gold_test_formatted_prompt1", format_input_text_prompt1, True),
    ("Gold_test_formatted_prompt2", format_input_text_prompt2, False),
]:
    formatted, errors = process_gold_test(format_fn, use_system_message=use_system)
    out_path = OUT_DIR / f"{name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for entry in formatted:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"{name}.jsonl : {len(formatted)} entrées -> {out_path}")
    if errors:
        print(f"  Erreurs: {len(errors)}", errors[:3])

print("\nTerminé.")

Gold_test_formatted_prompt1.jsonl : 500 entrées -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot/Gold_test_formatted_prompt1.jsonl
Gold_test_formatted_prompt2.jsonl : 500 entrées -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot/Gold_test_formatted_prompt2.jsonl

Terminé.


## 4. Vérification (aperçu d'une entrée)

## 5. Dataset few-shot (prompt avec exemples inclus)

Chaque ligne du JSONL contient le **prompt complet** : message system (optionnel) + **exemples few-shot** (user/assistant tirés du train) + **exemple de test** (user + assistant = gold label). Les exemples sont choisis par couple **(type, section)** pour matcher le test.

In [32]:
def process_json_to_examples(input_path, format_fn):
    """Lit un JSON (train ou test), formate chaque entrée. Retourne une liste de dicts
    avec user_content, label, type, section_id (pour indexation few-shot)."""
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    examples = []
    for entry_id, entry in data.items():
        try:
            statement = entry.get("Statement", "")
            label = entry.get("Label", "")
            section_id = entry.get("Section_id", "")
            primary_id = entry.get("Primary_id", "")
            entry_type = entry.get("Type", "Single")
            primary_ct_data = load_ct_data(primary_id)
            if primary_ct_data is None:
                continue
            primary_evidence = extract_evidence(
                primary_ct_data, section_id, entry.get("Primary_evidence_index", [])
            )
            secondary_evidence = None
            if entry_type == "Comparison":
                secondary_ct_data = load_ct_data(entry.get("Secondary_id", ""))
                if secondary_ct_data is None:
                    continue
                secondary_evidence = extract_evidence(
                    secondary_ct_data, section_id, entry.get("Secondary_evidence_index", [])
                )
            user_content = format_fn(statement, primary_evidence, secondary_evidence)
            examples.append({
                "user_content": user_content,
                "label": label,
                "type": entry_type,
                "section_id": section_id,
            })
        except Exception:
            continue
    return examples

TRAIN_JSON = NLI4CT_ROOT / "train.json"
train_examples_p1 = process_json_to_examples(TRAIN_JSON, format_input_text_prompt1) if TRAIN_JSON.exists() else []
train_examples_p2 = process_json_to_examples(TRAIN_JSON, format_input_text_prompt2) if TRAIN_JSON.exists() else []
print("Train: ", len(train_examples_p1), "exemples (Prompt 1)")

Train:  1700 exemples (Prompt 1)


In [33]:
from collections import defaultdict

def build_fewshot_index(examples):
    """Indexe les exemples par (type, section_id), puis par label."""
    index = defaultdict(lambda: {"Entailment": [], "Contradiction": []})
    for ex in examples:
        key = (ex["type"], ex["section_id"])
        if ex["label"] in index[key]:
            index[key][ex["label"]].append(ex)
    return index

def make_fewshot_messages(test_ex, shot_examples, use_system_message=False):
    """Construit la liste messages: [system?] + shots (user, asst) + test (user, asst)."""
    messages = []
    if use_system_message:
        messages.append({"role": "system", "content": SYSTEM_MSG_FEWSHOT})
    for shot in shot_examples:
        messages.append({"role": "user", "content": shot["user_content"]})
        messages.append({"role": "assistant", "content": shot["label"]})
    messages.append({"role": "user", "content": test_ex["user_content"]})
    messages.append({"role": "assistant", "content": test_ex["label"]})
    return messages

# Nombre d'exemples few-shot par label (1 Entailment + 1 Contradiction = 2 shots par défaut)
N_PER_LABEL = 2

In [34]:
import random

def select_shots(index, type_, section_id, n_per_label=1, rng=None):
    """Sélectionne n_per_label exemples Entailment et n_per_label Contradiction pour (type_, section_id)."""
    if rng is None:
        rng = random.Random(42)
    key = (type_, section_id)
    shots = []
    for label in ["Entailment", "Contradiction"]:
        pool = index[key].get(label, [])
        if not pool:
            continue
        k = min(n_per_label, len(pool))
        shots.extend(rng.sample(pool, k))
    rng.shuffle(shots)  # mélanger pour ne pas toujours avoir E puis C
    return shots

# Génération des deux jeux few-shot (Prompt 1 et Prompt 2)
rng = random.Random(42)
for name, train_examples, format_fn, use_system in [
    ("Gold_test_fewshot_prompt1", train_examples_p1, format_input_text_prompt1, True),
    ("Gold_test_fewshot_prompt2", train_examples_p2, format_input_text_prompt2, True),
]:
    if not train_examples:
        print(f"{name}: train.json introuvable, on ignore.")
        continue
    index = build_fewshot_index(train_examples)
    test_examples = process_json_to_examples(GOLD_TEST_JSON, format_fn)
    out_path = OUT_DIR / f"{name}.jsonl"
    count_ok = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for test_ex in test_examples:
            shots = select_shots(index, test_ex["type"], test_ex["section_id"], n_per_label=N_PER_LABEL, rng=rng)
            messages = make_fewshot_messages(test_ex, shots, use_system_message=use_system)
            f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
            count_ok += 1
    print(f"{name}.jsonl : {count_ok} lignes -> {out_path}")
print("Terminé.")

Gold_test_fewshot_prompt1.jsonl : 500 lignes -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot/Gold_test_fewshot_prompt1.jsonl
Gold_test_fewshot_prompt2.jsonl : 500 lignes -> /Users/lubin/Documents/NLI_Finetuning/NLI4CT/results/Fewshot/Gold_test_fewshot_prompt2.jsonl
Terminé.


In [35]:
# Aperçu d'une ligne few-shot (Prompt 1)
with open(OUT_DIR / "Gold_test_fewshot_prompt1.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(f.readline())
msgs = first["messages"]
print("Nombre de messages:", len(msgs))
print("Rôles:", [m["role"] for m in msgs])
print("Dernier user (début):", msgs[-2]["content"][:200], "...")
print("Dernier assistant (gold):", msgs[-1]["content"])

Nombre de messages: 11
Rôles: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']
Dernier user (début): PREMISE: Exclusion Criteria:
  History of bilateral mastectomy, osteoporosis or renal impairment.

Exclusion Criteria:
  Severe claustrophobia

HYPOTHESIS: Women suffering from both claustrophobia and ...
Dernier assistant (gold): Contradiction


In [36]:
formatted_p1, _ = process_gold_test(format_input_text_prompt1, use_system_message=True)
if formatted_p1:
    print("Exemple Prompt 1 (Gold test):")
    print(json.dumps(formatted_p1[0], indent=2, ensure_ascii=False)[:1200], "...")

Exemple Prompt 1 (Gold test):
{
  "messages": [
    {
      "role": "system",
      "content": "Classify the relationship between the premise and the hypothesis. Respond with only one word: 'Entailment' or 'Contradiction'."
    },
    {
      "role": "user",
      "content": "PREMISE: Exclusion Criteria:\n  History of bilateral mastectomy, osteoporosis or renal impairment.\n\nExclusion Criteria:\n  Severe claustrophobia\n\nHYPOTHESIS: Women suffering from both claustrophobia and IBS or not eligible for either the primary trial or the secondary trial."
    },
    {
      "role": "assistant",
      "content": "Contradiction"
    }
  ]
} ...
